# 01 — Dataset Exploration
Visualise a 4-band GeoTIFF scene and its ground-truth mask before training.

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio

from src.preprocessing.preprocess import (
    read_multiband_geotiff,
    read_mask_geotiff,
    apply_clahe_per_band,
    tile_image_and_mask,
)

In [ ]:
# ── Change these paths to your actual scene files ──────────────────────────
IMAGE_PATH = Path('../data/raw/scene.tif')
MASK_PATH  = Path('../data/raw/scene_mask.tif')

image, profile = read_multiband_geotiff(IMAGE_PATH)
mask = read_mask_geotiff(MASK_PATH)

print(f'Image shape : {image.shape}  dtype={image.dtype}')
print(f'Mask shape  : {mask.shape}   unique={np.unique(mask)}')
print(f'CRS         : {profile["crs"]}')
print(f'Transform   : {profile["transform"]}')

In [ ]:
CLASS_COLORS = {0: 'grey', 1: 'white', 2: 'navy'}
CLASS_NAMES  = {0: 'Background', 1: 'Cloud', 2: 'Shadow'}

# Gamma-correct RGB for display
rgb_display = np.power(np.clip(image[:, :, :3], 0, 1), 0.5)

# Build colour mask
mask_rgb = np.zeros((*mask.shape, 3))
palette = {0: [0.5,0.5,0.5], 1: [1,1,1], 2: [0.12,0.24,0.47]}
for cls_id, colour in palette.items():
    mask_rgb[mask == cls_id] = colour

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb_display)
axes[0].set_title('RGB Preview (bands 1–3)')
axes[0].axis('off')

axes[1].imshow(image[:, :, 3], cmap='gray')
axes[1].set_title('NIR Band (band 4)')
axes[1].axis('off')

axes[2].imshow(mask_rgb)
legend = [mpatches.Patch(color=c, label=CLASS_NAMES[i]) for i, c in palette.items()]
axes[2].legend(handles=legend, loc='lower right')
axes[2].set_title('Ground Truth Mask')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Class distribution ─────────────────────────────────────────────────────
total = mask.size
for cls_id, name in CLASS_NAMES.items():
    count = int(np.sum(mask == cls_id))
    print(f'{name:12s}: {count:>10,} px  ({100*count/total:.2f} %)')

In [ ]:
# ── CLAHE enhancement comparison ───────────────────────────────────────────
image_clahe = apply_clahe_per_band(image)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(np.power(image[:, :, :3], 0.5))
axes[0].set_title('Before CLAHE')
axes[0].axis('off')
axes[1].imshow(np.power(image_clahe[:, :, :3], 0.5))
axes[1].set_title('After CLAHE')
axes[1].axis('off')
plt.suptitle('CLAHE Contrast Enhancement (RGB display only)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Patch tiling preview ───────────────────────────────────────────────────
img_patches, mask_patches = tile_image_and_mask(
    image_clahe, mask, patch_size=256, overlap=0.25
)
print(f'Total patches: {len(img_patches)}')

# Show first 6 patches
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    axes[0, i].imshow(np.power(img_patches[i][:, :, :3], 0.5))
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Patch {i}')

    mp_rgb = np.zeros((*mask_patches[i].shape, 3))
    for cls_id, colour in palette.items():
        mp_rgb[mask_patches[i] == cls_id] = colour
    axes[1, i].imshow(mp_rgb)
    axes[1, i].axis('off')
    axes[1, i].set_title(f'Mask {i}')

plt.suptitle('Sample Patches (top: image, bottom: mask)')
plt.tight_layout()
plt.show()